# TODO:

Politics / Mentions: two promising wallets
0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8	
0x0cb10c40b0776e9ee8cef970af85724654dda76c


# Wallet Strategy Selection

Unified selection stage: runs **all wallet-selection methods** and saves each
as a named `WalletSelectionStrategy` artifact in the workspace.  The backtest
stage loads these artifacts directly — no strategy logic is re-built there.

## Methods included
| id suffix | description |
|-----------|-------------|
| `skill_sweep` → `quality_core` | top-N by best skill metric (grid-searched on train-a→train-b) |
| `cohort_early_entry` | wallets that enter markets early |
| `cohort_smooth_pnl` | wallets with high PnL relative to volatility |
| `volatility` | volatility-filtered wallets (from the profitable_wallet_analysis path) |

Each method produces **three trigger variants**:
- `score_threshold`  — composite signal score ≥ calibrated threshold (Kelly sizing)
- `all_open_buys`    — every open-buy event (fixed stake baseline)
- `copy_triggers`    — every trade (open/add/close/reduce), tight slippage


In [76]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path
import json

import numpy as np
import pandas as pd
import pyarrow.dataset as ds

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.2f}'.format

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration

In [77]:
# ── configuration ─────────────────────────────────────────────────────────────
RECENCY_DAYS     = 90

PRICE_BUCKET_BINS   = [0.0, 0.1, 0.25, 0.4, 0.6, 0.75, 0.9, 1.0]
PRICE_BUCKET_LABELS = [
    "0.00-0.10", "0.10-0.25", "0.25-0.40", "0.40-0.60",
    "0.60-0.75", "0.75-0.90", "0.90-1.00",
]

TRADES_DIR    = Path("../../data/polygon_trades_processed")
WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

dataset = ds.dataset(TRADES_DIR, format="parquet")
DATASET_COLUMNS        = set(dataset.schema.names)
TRIGGER_TX_HASH_COLUMN = (
    "transaction_hash" if "transaction_hash" in DATASET_COLUMNS
    else ("tx_hash" if "tx_hash" in DATASET_COLUMNS else None)
)

METRICS_A_PATH          = WORKSPACE_DIR / "wallet_metrics_train_a.parquet"
METRICS_B_PATH          = WORKSPACE_DIR / "wallet_metrics_train_b.parquet"
METRICS_FULL_PATH       = WORKSPACE_DIR / "wallet_metrics_full_train.parquet"
METRICS_TEST_PATH       = WORKSPACE_DIR / "wallet_metrics_test.parquet"
CALIBRATION_SIGNALS_PATH = WORKSPACE_DIR / "signal_events_train_b.parquet"
TEST_SIGNALS_PATH       = WORKSPACE_DIR / "signal_events_test.parquet"


## Derive train/test dates from data

In [78]:
# ── derive train/test boundary from is_train flag ───────────────────────────
_sample = pd.read_parquet(sorted(TRADES_DIR.glob("*.parquet"))[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN   = _sample.loc[_sample["is_train"], "dt"].max().date()
TRAIN_START_DATE = _sample.loc[_sample["is_train"], "dt"].min().date()
# Split train_a / train_b at the trade-count median of training rows so that
# both halves always contain roughly equal numbers of trades even when the
# data is unevenly distributed over calendar time.
_train_rows = _sample.loc[_sample["is_train"]].sort_values("dt")
TRAIN_A_END_DATE = _train_rows["dt"].quantile(0.5, interpolation="nearest").date()
del _sample, _train_rows
print(f"END_DATE_TRAIN={END_DATE_TRAIN}  TRAIN_A_END_DATE={TRAIN_A_END_DATE}")


END_DATE_TRAIN=2026-02-15  TRAIN_A_END_DATE=2025-10-07


## Compute / load wallet skill metrics

In [79]:
from polymarket_analysis.wallet_selection.metrics import compute_wallet_skill_workspace

if all(p.exists() for p in [METRICS_A_PATH, METRICS_B_PATH, METRICS_FULL_PATH, METRICS_TEST_PATH]):
    train_a_metrics    = pd.read_parquet(METRICS_A_PATH)
    train_b_metrics    = pd.read_parquet(METRICS_B_PATH)
    full_train_metrics = pd.read_parquet(METRICS_FULL_PATH)
    test_metrics       = pd.read_parquet(METRICS_TEST_PATH)
else:
    train_a_metrics, train_b_metrics, full_train_metrics, test_metrics = (
        compute_wallet_skill_workspace(
            dataset,
            end_date_train=END_DATE_TRAIN,
            train_a_end_date=TRAIN_A_END_DATE,
            recency_days=RECENCY_DAYS,
        )
    )
    train_a_metrics.to_parquet(METRICS_A_PATH, index=False)
    train_b_metrics.to_parquet(METRICS_B_PATH, index=False)
    full_train_metrics.to_parquet(METRICS_FULL_PATH, index=False)
    test_metrics.to_parquet(METRICS_TEST_PATH, index=False)

pd.Series({
    "train_a_wallets":    len(train_a_metrics),
    "train_b_wallets":    len(train_b_metrics),
    "full_train_wallets": len(full_train_metrics),
    "test_wallets":       len(test_metrics),
}).to_frame("value")


,value
train_a_wallets,10964
train_b_wallets,10964
full_train_wallets,10964
test_wallets,10964


## Cohort selection sweep (skill path)

Grid-search over metrics × top-N using train-a → train-b persistence.

In [80]:
from polymarket_analysis.wallet_selection.selector import (
    CANDIDATE_METRICS,
    cohort_selection_sweep,
    select_wallets,
)

cohort_sweep = cohort_selection_sweep(train_a_metrics, train_b_metrics, CANDIDATE_METRICS)
cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge"], ascending=False
).head(20)


,metric,top_n,wallets_found_in_train_b,train_b_open_buy_trades,train_b_weighted_prob_edge,train_b_avg_prob_edge,train_b_avg_copy_roi_capped,train_b_total_copy_pnl_usdc,train_b_hit_rate
10,avg_copy_roi_capped,50,50,1813,0.07,0.12,1.52,103284.02,0.42
11,avg_copy_roi_capped,100,100,2695,0.07,0.10,1.12,143225.93,0.42
0,prob_edge_shrunk,50,50,4204,0.01,0.12,0.93,115130.45,0.47
22,roi_sharpe,200,200,3250,0.01,0.07,0.85,101101.91,0.70
12,avg_copy_roi_capped,200,200,5272,0.06,0.11,0.82,248959.70,0.48
1,prob_edge_shrunk,100,100,5418,0.03,0.11,0.78,201409.43,0.51
13,avg_copy_roi_capped,300,300,7046,0.06,0.07,0.76,384078.84,0.50
23,roi_sharpe,300,300,4467,0.01,0.07,0.70,117444.18,0.70
6,weighted_prob_edge_shrunk,100,100,2399,0.06,0.09,0.68,256835.63,0.47
21,roi_sharpe,100,100,2473,0.01,0.03,0.66,73063.49,0.78


In [81]:
# pick best metric / top-N
best_row = cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge", "train_b_open_buy_trades"],
    ascending=[False, False, False],
).iloc[0]
BEST_SELECTION_METRIC = best_row["metric"]
BEST_TOP_N            = int(best_row["top_n"])
{"best_metric": BEST_SELECTION_METRIC, "best_top_n": BEST_TOP_N}


{'best_metric': 'avg_copy_roi_capped', 'best_top_n': 50}

## Select wallets (skill path) + build cohorts

## Build wallet profiles and signal events

In [82]:
# from polymarket_analysis.signal.builder import (
#     build_wallet_profiles,
#     build_signal_events,
# )

# selected_wallet_profiles = build_wallet_profiles(
#     dataset, selected_wallets, period="full_train",
#     end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE
# )
# selected_wallet_profiles.to_parquet(
#     WORKSPACE_DIR / "selected_wallet_profiles_v2.parquet", index=False
# )

# # Force-regenerate signal caches: the schema now includes all event types
# # (open_buy, add_buy, close_sell, reduce_sell) plus copy_price/copy_market_key/
# # copy_token_winner columns. Delete stale parquets so they are always rebuilt.
# CALIBRATION_SIGNALS_PATH.unlink(missing_ok=True)
# TEST_SIGNALS_PATH.unlink(missing_ok=True)

# _, train_b_signals = build_signal_events(
#     dataset, selected_wallet_profiles, period="train_b",
#     end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
#     price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
#     tx_hash_column=TRIGGER_TX_HASH_COLUMN,
# )
# train_b_signals.to_parquet(CALIBRATION_SIGNALS_PATH, index=False)

# _, test_signals = build_signal_events(
#     dataset, selected_wallet_profiles, period="test",
#     end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
#     price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
#     tx_hash_column=TRIGGER_TX_HASH_COLUMN,
# )
# test_signals.to_parquet(TEST_SIGNALS_PATH, index=False)

# print(f"train_b signals: {len(train_b_signals):,}  test signals: {len(test_signals):,}")
# print(f"event types: {train_b_signals['event_type'].value_counts().to_dict()}")

## Calibrate signal scoring on train-B

In [83]:
# from polymarket_analysis.signal.scorer import (
#     build_calibration_tables,
#     apply_signal_score,
#     select_signal_threshold,
# )

# # Calibration tables are built from open_buy events only — the scoring model
# # is designed for open-buy conviction features. Non-open-buy rows get neutral
# # defaults and will not be used to calibrate the scorer.
# train_b_open_buys = train_b_signals[train_b_signals["event_type"] == "open_buy"].copy()

# price_table, consensus_table, global_edge = build_calibration_tables(train_b_open_buys)
# train_b_scored = apply_signal_score(train_b_open_buys, price_table, consensus_table)
# SIGNAL_THRESHOLD = select_signal_threshold(train_b_scored)
# print(f"Global edge: {global_edge:.4f}")
# print(f"Selected signal threshold: {SIGNAL_THRESHOLD:.2f}")

# # score distribution
# score_deciles = (
#     train_b_scored.assign(
#         score_decile=lambda df: pd.qcut(df["signal_score"], q=10, labels=False, duplicates="drop")
#     )
#     .groupby("score_decile")
#     .agg(
#         count=("signal_score", "size"),
#         avg_signal_score=("signal_score", "mean"),
#         avg_copy_roi_capped=("copy_roi_capped", "mean"),
#     )
#     .reset_index()
# )
# score_deciles

## Build wallet cohorts (skill path)

In [84]:
from polymarket_analysis.wallet_selection.selector import build_wallet_cohorts

# wallet_cohorts = build_wallet_cohorts(
#     full_train_metrics, train_b_open_buys, selected_wallets,
#     top_n=BEST_TOP_N,
# )
# {name: len(df) for name, df in wallet_cohorts.items()}


## Volatility-based wallet selection (second method)

Loads the full training set, applies the volatility filter, and stores the
result as an additional `WalletSelectionStrategy`.  The volatility wallet set
is added to `wallet_cohorts` so the strategy factory handles it uniformly.

In [85]:
len(df_full[df_full['dt'] >= pd.to_datetime("2026-03-31", utc=True)])

165

In [86]:
from polymarket_analysis.wallet_selection.volatility import (
    compute_wallet_metrics,
    filter_wallets_by_volatility,
)

from polymarket_analysis.data.data_catalogue import load_markets_processed
mdf = load_markets_processed()
mdf = mdf[
    ~(mdf['primary_tag'].isin(['Sports', 'Crypto']))
    & (mdf['winner_token_id'].notna())
    ].copy().reset_index(drop=True)


# Load full training trades for the volatility path
trade_files = sorted(TRADES_DIR.glob("*.parquet"))
df_full = pd.concat([pd.read_parquet(f) for f in trade_files], ignore_index=True)

df_full = df_full.merge(mdf, on="condition_id", how="inner")

df_full['outcome'] = df_full['outcome_x']
del df_full['outcome_x'], df_full['outcome_y']

df_full["dt"] = pd.to_datetime(df_full["dt"], utc=True)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_full.columns and "quantity" not in df_full.columns:
    df_full = df_full.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_full["usdc_amount"]      = df_full["usdc_amount"].astype(float)
df_full["final_value_usdc"] = df_full["final_value_usdc"].astype(float)
df_full["quantity"]         = df_full["quantity"].astype(float)

df_full["pnl"] = np.where(
    df_full["side"] == "BUY",
    df_full["final_value_usdc"] - df_full["usdc_amount"],
    df_full["usdc_amount"] - df_full["final_value_usdc"],
)
df_full["notional"] = np.where(
    df_full["side"] == "BUY",
    df_full["usdc_amount"],
    df_full["quantity"] * (1 - df_full["price"].astype(float)),
)
# ignore buys with large price, so we don't skew roi
# df_full = df_full[df_full['price'] <= 0.95]
df_slice = df_full[df_full["is_train"]].copy()

wallet_vol_train, _ = compute_wallet_metrics(df_slice)
print(len(wallet_vol_train))
# del df_full, df_slice

15720


In [87]:
wallet_vol_train['copyable_pnl_factor'] = np.clip(wallet_vol_train['copyable_pnl'] / wallet_vol_train['total_pnl'], 0, 1.0)
wallet_vol_train['copyable_roi'] = wallet_vol_train['average_roi'] * wallet_vol_train['copyable_pnl_factor']
wallet_vol_train.sort_values('copyable_roi', ascending=False, inplace=True)
wallet_vol_train.reset_index(drop=True, inplace=True)
# wallet_vol_train = wallet_vol_train[wallet_vol_train['copyable_roi'] > 0.05]
# wallet_vol_train = wallet_vol_train[wallet_vol_train['top_market_pnl_pct'] < 0.3]

In [329]:
wallet_cohorts = {}

In [330]:
print(len(wallet_vol_train))
wallet_vol_train.head()


15720


,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top_market_pnl_pct,median_roi,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi
0,0x62b61c25cd18c81d15867f62a6d708aff526e575,5.72,2.00,1.00,159.38,56.00,56.00,1.00,1.00,49.00,49.00,157.23,2.81,157.23,2.81,0.35,1.00,49.00
1,0x53e090f9f25364cd2bb0c60b9db18706f6fcb08e,16.70,2.00,1.00,405.34,25.87,25.87,1.00,1.00,49.00,49.00,401.03,15.50,401.03,15.50,0.06,1.00,49.00
2,0xbba9a972d6f3e8fb79792dad96fe186b55f59051,NaN,1.00,1.00,1.00,124.00,39.68,1.00,1.00,124.00,124.00,0.00,0.00,0.00,0.00,124.00,0.32,39.68
3,0xce68fe75d432985c5a83f2b8bf78a5da9c8d3cf0,NaN,1.00,1.00,1.86,67.05,67.05,1.00,1.00,36.04,36.04,0.00,0.00,0.00,0.00,36.04,1.00,36.04
4,0x6ac7558bf37ded622fa1098265b42e8cc8cd735f,NaN,1.00,1.00,1.57,110.43,41.10,1.00,1.00,70.43,70.43,0.00,0.00,0.00,0.00,70.43,0.37,26.21


In [331]:
# filtered_wallets_vol = filter_wallets_by_volatility(
#     wallet_vol_train,
#     min_buckets=20,
#     max_top5_pnl_pct=.6,
#     max_top_market_pnl_pct=0.5,
# )

# filtered_wallets_vol = wallet_vol_train.copy()

# filtered_wallets_vol = (
#     filtered_wallets_vol[
#         (filtered_wallets_vol['average_roi'] >= 0.04)
#         & (filtered_wallets_vol['copyable_pnl'] / filtered_wallets_vol['total_pnl'] >= 0.5)
# ].sort_values("pnl_volatility", ascending=True).head(100)
# )

# print(f"Volatility-filtered wallets: {len(filtered_wallets_vol)}")

# # Build wallet_quality based on total_pnl rank (higher = better)
# filtered_wallets_vol = filtered_wallets_vol.copy().reset_index(drop=True)
# filtered_wallets_vol["wallet_quality"] = filtered_wallets_vol["total_pnl"].rank(
#     method="first", pct=True
# )

# Add as additional cohort (uses only train-b signals for trigger calibration)
# We intersect with wallets that have signals to ensure coverage
# vol_wallets_with_signals = set(train_b_signals["wallet"]).intersection(
#     set(filtered_wallets_vol["wallet"])
# )
# [
#     filtered_wallets_vol["wallet"].isin(vol_wallets_with_signals)
# ][["wallet", "wallet_quality"]].copy().reset_index(drop=True)

In [332]:
wallet_vol_train.sort_values("copyable_pnl", ascending=False).head(10)

,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top_market_pnl_pct,median_roi,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi
5376,0x17db3fcd93ba12d38382a0cade24b200185c5f6d,20.42,655.00,42.00,3638656.20,1304494.87,501531.92,0.45,0.33,0.27,0.35,62723.87,0.05,23756.96,0.05,0.36,0.38,0.13
3805,0x2a21fec355312b609b9ac97ce4eda3e93ec3d307,18.78,61.00,17.00,1559733.52,268316.85,240110.13,0.86,0.30,0.11,0.32,21166.73,0.08,21001.44,0.09,0.17,0.89,0.29
2861,0x1f1d38f54a615d1c43c77e3b109bd01f56e163d9,20.33,577.00,17.00,769614.27,688774.49,222966.54,0.57,0.66,-1.00,1.39,114228.78,0.17,47935.04,0.21,0.89,0.32,0.45
5068,0x551e72eda42a5ab39d6d78239a1d9bbb5db6b0e0,10.95,2329.00,313.00,3643298.75,282061.18,154224.66,0.83,0.62,0.02,0.29,109667.48,0.39,35283.79,0.23,0.08,0.55,0.16
10825,0xbacd00c9080a82ded56f504ee8810af732b0ab35,4.10,46103.00,1059.00,22196501.74,645638.20,140370.19,0.24,0.16,-1.00,-0.10,122370.76,0.19,77297.94,0.55,0.03,0.22,-0.02
5558,0x05e26c775ecfe91b897d47f134c1bf5900ca6e12,26.40,677.00,50.00,1490203.14,426677.40,137219.36,1.10,0.58,0.14,0.38,116315.41,0.27,96660.99,0.70,0.29,0.32,0.12
2265,0xbad457dc633bbb7b6cbe09dd5867a5e8e597acd7,146.12,400.00,5.00,340760.10,752200.10,131176.42,0.79,0.99,2.45,3.56,1100.00,0.00,570.47,0.00,2.21,0.17,0.62
4159,0x8861f0bb5e0c19474ba73beeadc13ed8915beed6,16.03,325.00,17.00,1361177.08,556342.53,129122.41,0.48,0.45,0.37,1.05,25732.59,0.05,499.11,0.00,0.41,0.23,0.24
11913,0xecb14ac6e9ca447ce2f2912e6217b43d7b655da3,16.37,4729.00,399.00,11468383.33,184403.15,125135.36,0.93,0.49,0.01,-0.07,133224.19,0.72,21567.14,0.17,0.02,0.68,-0.05
5876,0xbaa2bcb5439e985ce4ccf815b4700027d1b92c73,12.84,13025.00,483.00,12337856.08,358060.21,120181.19,0.78,0.21,0.04,0.31,156872.42,0.44,62355.67,0.52,0.03,0.34,0.10


In [578]:
# wallet_cohorts['multi_filter'] = wallet_vol_train[
#     (wallet_vol_train['copyable_roi'] >= 0.04)
#     & (wallet_vol_train['num_buckets'] >= 20)
#     &(wallet_vol_train['copyable_pnl'] <= wallet_vol_train['total_pnl'])
#     &(wallet_vol_train['copyable_pnl_factor'] >=0.4)
#     # & (wallet_vol_train['median_roi'] >= 0)
#     & (wallet_vol_train['max_drawdown_to_pnl'] <= 0.3)
#     & (wallet_vol_train['max_copyable_drawdown_to_copyable_pnl'] <= 0.4)
#     # & (wallet_vol_train['num_markets'] >= 100)
#     & (wallet_vol_train['top_market_pnl_pct'] < 0.4)
#     & (wallet_vol_train['top5_pnl_pct'] < 0.5)
#     # & (wallet_vol_train['total_pnl'] < 10_000)
# ].sort_values("copyable_roi", ascending=False).copy().reset_index(drop=True)

wallet_cohorts['multi_filter'] = wallet_vol_train[
    # (wallet_vol_train['copyable_roi'] >= 0.02)
    # & (wallet_vol_train['copyable_pnl'] <= wallet_vol_train['total_pnl'])
    (wallet_vol_train['max_drawdown_to_pnl'] <= 0.3)
    # & (wallet_vol_train['max_drawdown_to_pnl'] >= 0.1)
    # & (wallet_vol_train['max_copyable_drawdown_to_copyable_pnl'] <= 0.4)
    & (wallet_vol_train['num_buckets'] >= 20)
    & (wallet_vol_train['top_market_pnl_pct'] < 0.3)
    & (wallet_vol_train['top5_pnl_pct'] < 0.5)
    # & (wallet_vol_train['copyable_pnl_factor'] >= 0.1)
    # & (wallet_vol_train['total_pnl'] > 10_000)
].sort_values("copyable_roi", ascending=False).copy().reset_index(drop=True)

len(wallet_cohorts['multi_filter'])

218

In [583]:
wallet_vol_train.head()

,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top_market_pnl_pct,median_roi,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi
0,0x62b61c25cd18c81d15867f62a6d708aff526e575,5.72,2.00,1.00,159.38,56.00,56.00,1.00,1.00,49.00,49.00,157.23,2.81,157.23,2.81,0.35,1.00,49.00
1,0x53e090f9f25364cd2bb0c60b9db18706f6fcb08e,16.70,2.00,1.00,405.34,25.87,25.87,1.00,1.00,49.00,49.00,401.03,15.50,401.03,15.50,0.06,1.00,49.00
2,0xbba9a972d6f3e8fb79792dad96fe186b55f59051,NaN,1.00,1.00,1.00,124.00,39.68,1.00,1.00,124.00,124.00,0.00,0.00,0.00,0.00,124.00,0.32,39.68
3,0xce68fe75d432985c5a83f2b8bf78a5da9c8d3cf0,NaN,1.00,1.00,1.86,67.05,67.05,1.00,1.00,36.04,36.04,0.00,0.00,0.00,0.00,36.04,1.00,36.04
4,0x6ac7558bf37ded622fa1098265b42e8cc8cd735f,NaN,1.00,1.00,1.57,110.43,41.10,1.00,1.00,70.43,70.43,0.00,0.00,0.00,0.00,70.43,0.37,26.21


In [624]:
df_full[
    (df_full["dt"] >= split_dt)
    & (df_full["side"] == 'BUY')
    & (df_full['primary_tag'] == 'Politics')
    & (~df_full['condition_id'].isin(mention_markets))
    & (df_full['wallet'].isin(wallet_cohorts['multi_filter']['wallet']))][['pnl', 'copyable_pnl']].sum()
    # ].groupby('wallet')[['pnl', 'copyable_pnl']].sum().sort_values(ascending=False, key=abs, by='copyable_pnl').head(30)

pnl            2480195.64
copyable_pnl    971300.63
dtype: float64

In [585]:
wallet_cohorts['multi_filter'].sort_values("copyable_pnl", ascending=False).head(20)

,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top_market_pnl_pct,median_roi,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi
161,0xbacd00c9080a82ded56f504ee8810af732b0ab35,4.10,46103.00,1059.00,22196501.74,645638.20,140370.19,0.24,0.16,-1.00,-0.10,122370.76,0.19,77297.94,0.55,0.03,0.22,-0.02
182,0x7c3db723f1d4d8cb9c550095203b686cb11e5c6b,4.02,30334.00,2076.00,27146811.52,318377.42,111467.32,0.29,0.15,0.00,-0.10,47413.59,0.15,31406.25,0.28,0.01,0.35,-0.04
130,0xc6587b11a2209e46dfe3928b31c5514a8e33b784,10.90,1878.00,192.00,29029528.92,702646.16,92649.78,0.38,0.19,0.01,-0.04,91701.06,0.13,30140.95,0.33,0.02,0.13,-0.01
39,0x6bab41a0dc40d6dd4c1a915b8c01969479fd1292,4.69,3287.00,521.00,8600376.40,311067.18,90531.54,0.21,0.11,0.01,0.12,56515.67,0.18,19993.69,0.22,0.04,0.29,0.03
94,0x44c1dfe43260c94ed4f1d00de2e1f80fb113ebc1,7.27,5448.00,567.00,24075463.51,458591.40,78321.13,0.34,0.16,0.01,0.01,133275.67,0.29,96349.38,1.23,0.02,0.17,0.00
35,0x68c24bf4a8ad4d79a6fe4b8eec6f93a02dfd1711,10.28,4624.00,897.00,2517113.49,262084.12,54959.65,0.35,0.09,0.06,0.19,51778.22,0.20,21708.16,0.39,0.10,0.21,0.04
20,0xdfe3fedc5c7679be42c3d393e99d4b55247b73c4,10.74,1543.00,136.00,2301560.26,272752.75,52629.68,0.20,0.21,0.20,0.39,33000.00,0.12,2040.00,0.04,0.12,0.19,0.08
196,0x7131cb618fb195c70e8e629c45199096833c29cc,5.76,1371.00,127.00,3388042.04,132844.50,49064.84,0.45,0.16,0.04,-0.14,22241.91,0.17,20414.20,0.42,0.04,0.37,-0.05
47,0x5739ddf8672627ce076eff5f444610a250075f1a,4.82,2092.00,84.00,2079415.33,240488.96,45422.06,0.16,0.14,0.15,0.12,46039.12,0.19,34489.51,0.76,0.12,0.19,0.02
50,0x24c8cf69a0e0a17eee21f69d29752bfa32e823e1,3.48,19345.00,606.00,32796941.30,237555.34,44712.50,0.38,0.20,0.00,0.12,46781.11,0.20,48161.93,1.08,0.01,0.19,0.02


In [586]:
df_slice.head()

,wallet,condition_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,trade_pnl,...,avail_copy_total_vol,avail_copy_count,end_date_iso,question,tags,primary_tag,winner_token_id,outcome,pnl,notional
0,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0x072acaa7dd352c795948d495e019eec539539d36698225ab98b2f55c692d1b0c,2025-01-11 15:56:11+00:00,BUY,1633.00,1633.00,0.87,1418.29,0.00,-1418.29,...,-0.00,0.00,2025-01-20T00:00:00Z,Will Biden pardon Adam Schiff?,"[Politics, Biden, Hunter, Creators, zerohedge, pardon]",Politics,110508738387624671992646538975675731011839126477460430600317798022691903999417,No,-1418.29,1418.29
1,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0x072acaa7dd352c795948d495e019eec539539d36698225ab98b2f55c692d1b0c,2025-01-11 18:33:14+00:00,BUY,2350.00,717.00,0.87,623.07,0.00,-623.07,...,148.49,2.00,2025-01-20T00:00:00Z,Will Biden pardon Adam Schiff?,"[Politics, Biden, Hunter, Creators, zerohedge, pardon]",Politics,110508738387624671992646538975675731011839126477460430600317798022691903999417,No,-623.07,623.07
2,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0x072acaa7dd352c795948d495e019eec539539d36698225ab98b2f55c692d1b0c,2025-01-11 18:34:12+00:00,BUY,2400.90,50.90,0.87,44.23,0.00,-44.23,...,148.49,2.00,2025-01-20T00:00:00Z,Will Biden pardon Adam Schiff?,"[Politics, Biden, Hunter, Creators, zerohedge, pardon]",Politics,110508738387624671992646538975675731011839126477460430600317798022691903999417,No,-44.23,44.23
3,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0x072acaa7dd352c795948d495e019eec539539d36698225ab98b2f55c692d1b0c,2025-01-11 18:34:32+00:00,BUY,3493.34,1092.44,0.86,944.96,0.00,-944.96,...,-0.00,0.00,2025-01-20T00:00:00Z,Will Biden pardon Adam Schiff?,"[Politics, Biden, Hunter, Creators, zerohedge, pardon]",Politics,110508738387624671992646538975675731011839126477460430600317798022691903999417,No,-944.96,944.96
4,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0x072acaa7dd352c795948d495e019eec539539d36698225ab98b2f55c692d1b0c,2025-01-11 18:48:32+00:00,BUY,4400.35,907.01,0.86,784.56,0.00,-784.56,...,-0.00,0.00,2025-01-20T00:00:00Z,Will Biden pardon Adam Schiff?,"[Politics, Biden, Hunter, Creators, zerohedge, pardon]",Politics,110508738387624671992646538975675731011839126477460430600317798022691903999417,No,-784.56,784.56


In [625]:
wallets = set(wallet_cohorts['multi_filter']['wallet'])

df = df_full[~df_full["is_train"]].copy()
print(len(df))

df_wallets = df[
    (df['wallet'].isin(wallets))
    & (df['end_date_iso'] >= split_dt.isoformat())
    ].copy()
print(len(df_wallets))

df = df_wallets.groupby('condition_id').agg(
    pnl=('pnl', 'sum'),
    notional=('notional', 'sum'),
    sell_count=('side', lambda x: (x == 'SELL').sum()),
    buy_count=('side', lambda x: (x == 'BUY').sum()),
    wallets=('wallet', lambda x: x.nunique()),
).sort_values(by="pnl", key=np.abs, ascending=False).merge(mdf, on="condition_id", how="left")

len(df)
df.sort_values(by='wallets', ascending=False).head(10)

964138
263419


,condition_id,pnl,notional,sell_count,buy_count,wallets,end_date_iso,question,tags,primary_tag,winner_token_id,outcome
0,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3ac4919650de9eeb94ed8,441920.18,31079511.56,2435,7242,82,2026-02-28T00:00:00Z,Khamenei out as Supreme Leader of Iran by February 28?,"[Politics, Iran, Middle East, Geopolitics, World]",Politics,39317885422026394259056328144566743331998444273202427934141325790266108570112,Yes
1,0x61ce3773237a948584e422de72265f937034af418a8b703e3a860ea62e59ff36,178819.37,3766942.91,2261,3045,75,2026-03-31T00:00:00Z,Will the Iranian regime fall by March 31?,"[Politics, Iran, Middle East, Israel, Geopolitics, World, Khamenei, Reza Pahlavi, Iran Regime]",Politics,101443205557953769944584140209356208074985603773882357999963509274129845972413,No
13,0x70909f0ba8256a89c301da58812ae47203df54957a07c7f8b10235e877ad63c2,29960.73,3303012.97,1289,2246,68,2026-03-31T00:00:00Z,Khamenei out as Supreme Leader of Iran by March 31?,"[Politics, Iran, Middle East, Israel, Geopolitics, World, Khamenei, Parent For Derivative]",Politics,70771354585365381988139008309072205730081182435161568795508496003376222185889,Yes
5,0x24fb7c2d95c93a68018e6c4a90d88043bb67d32fd1454924cef8ebdd550228f3,61312.25,17536684.51,775,2289,47,2026-03-31T00:00:00Z,Will Israel launch a major ground offensive in Lebanon by March 31?,"[Politics, Middle East, Israel, Lebanon, Geopolitics, World, Israel x Iran]",Politics,62174615336627888814453166657652087168672936561990669762061326057126859157348,Yes
24,0x30a892ea01a40602dd58029243f165a97ce22dbfeabf427f86d705ae650fad4b,11146.98,463433.90,440,676,43,2026-03-01T00:00:00Z,US forces enter Iran by March 14?,"[Iran, Trump, Middle East, Geopolitics, Military Strikes]",Iran,85520490413572360235058851450781930532680155921712950189237409407488519651948,No
11,0xd25b2fc3a916b317d71b398c5d0f81ad33fe1da6b56d6c9f717332d31584504e,38629.03,807016.84,581,871,36,2026-03-31T00:00:00Z,Will Reza Pahlavi enter Iran by March 31?,"[Iran, Middle East, Israel, Geopolitics, World, shah, Reza Pahlavi, Iran Regime]",Iran,86620530406157559149607427304423476709238424342859385697887050586116236047982,No
40,0x4befb52f7cd2b606c63731ad74e677325ff0ed0f48f1104552a34f0f54619ea3,6074.28,477043.55,470,1532,35,2026-03-31T00:00:00Z,Kharg Island no longer under Iranian control by March 31?,"[Politics, Iran, Strait of Hormuz, Geopolitics, Khamenei, Iran Regime, Mojtaba Khamenei, Kharg I...",Politics,46118971458650555165410440115606491822058595161731979755989518360335087029561,No
8,0xb3ebf217cf2f393a66030c072b04b893268506923e01b23f1bcf3504c3d319c2,48139.24,437650.58,841,1182,35,2026-02-28T00:00:00Z,Iran Strike on Israel by February 28?,"[Politics, Iran, Middle East, Israel, Geopolitics, Khamenei]",Politics,40785716512154576515459243689202568309525193875518923120378208451954796944952,Yes
6,0xc49142a0a94b68377bbc3f2e7f7f53a400274e2dcd4536cfa343be55a3c1fdf8,60396.94,155156.56,804,956,34,2026-03-31T00:00:00Z,Will another country strike Iran by March 31?,"[Politics, Iran, Israel, Geopolitics, Military Strikes]",Politics,43844893274652694909493529099004269415454316401232185138819429254111040362035,No
45,0xf94145fa5716d4bfd2948ce5eef2954dcb560516ccbba5e8b68059027e431cdc,5580.95,208529.78,272,324,33,2026-03-03T00:00:00Z,US forces enter Iran by March 7?,"[Iran, Trump, Middle East, Geopolitics, Military Strikes]",Iran,17667954885432047474600988169826078422913327586493371051607749430874729004890,No


In [630]:
MARKET = '0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20'

market_def = mdf.loc[mdf['condition_id'] == MARKET].head(1).squeeze()
print(f"Market: {market_def}")

dfc = df_full[(df_full['condition_id'] == MARKET)
              & (df_full['wallet'].isin(wallets))].copy()
# dfc['bucket'] = dfc['dt'].dt.floor('1h')


dfc = dfc.groupby(['dt', 'wallet', 'side', 'outcome']).agg( 
        pnl=('pnl', 'sum'),
        notional=('notional', 'sum'),
        quantity=('quantity', 'sum'),
        position=('position', 'last'),
        avg_price=('price', lambda x: (x * dfc.loc[x.index, 'quantity']).sum() / dfc.loc[x.index, 'quantity'].sum()),
        copyable_qty=('copyable_qty', 'sum'),
        copyable_pnl=('copyable_pnl', 'sum'),
)[['pnl', 'position', 'notional', 'quantity', 'avg_price', 'copyable_qty', 'copyable_pnl']].reset_index().sort_values(['dt', 'wallet', 'side', 'outcome'])

Market: condition_id                   0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20
end_date_iso                                                                 2026-01-31T00:00:00Z
question                                                    US strikes Iran by February 28, 2026?
tags                [Politics, Iran, Middle East, Geopolitics, World, Parent For Derivative, HFC]
primary_tag                                                                              Politics
winner_token_id    110790003121442365126855864076707686014650523258783405996925622264696084778807
outcome                                                                                       Yes
Name: 59618, dtype: object


In [641]:
dfc[(dfc['wallet'] == '0xc02147dee42356b7a4edbb1c35ac4ffa95f61fa8')
    & (dfc['quantity'] >= 1) ]['pnl']

0         -11.04
1        -117.39
2          -5.87
4          -3.37
5        -294.33
6        -432.00
165      -172.69
190      -510.00
233      -230.00
759       360.00
761       292.00
2575     -740.00
3480     2866.99
3777     2490.00
3868     4450.00
4344     4550.00
4528   -11959.99
4602    -4650.00
4654   -13020.00
4658    -3680.00
4753     -413.39
4754      -68.33
4755     -218.66
4756      -24.60
4757     -915.01
6276      200.00
6279      158.90
6295       90.00
6335       87.00
6338      100.00
6344        3.00
6345       49.00
6371       10.00
Name: pnl, dtype: float64

In [631]:
# ── positions per wallet + sum, dual-axis (position + price) ──────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if(len(dfc) != 0):

    # ── 1. build per-bucket price series for YES (volume-weighted avg)
    _yes = dfc[dfc['outcome'] == 'Yes'].copy()
    _yes['_wv'] = _yes['avg_price'] * _yes['quantity']
    price_ts = (
        _yes.groupby('dt')[['_wv', 'quantity']].sum()
        .rename(columns={'quantity': '_qty'})
        .assign(vwap=lambda d: d['_wv'] / d['_qty'])[['vwap']]
        .reset_index()
        .sort_values('dt')
    )

    # ── 2. net YES position per wallet per timestamp (YES pos - NO pos)
    # position is cumulative after each trade; take last value per (dt, wallet, outcome)
    pos_per_wallet = (
        dfc
        .groupby(['dt', 'wallet', 'outcome'])['position']
        .last()
        .reset_index()
    )
    pos_yes = pos_per_wallet[pos_per_wallet['outcome'] == 'Yes'].rename(columns={'position': 'pos_yes'})[['dt', 'wallet', 'pos_yes']]
    pos_no  = pos_per_wallet[pos_per_wallet['outcome'] == 'No' ].rename(columns={'position': 'pos_no' })[['dt', 'wallet', 'pos_no' ]]
    net_pos = (
        pos_yes
        .merge(pos_no, on=['dt', 'wallet'], how='outer')
        .fillna(0)
    )
    net_pos['net'] = net_pos['pos_yes'] - net_pos['pos_no']
    net_pos = net_pos.sort_values(['wallet', 'dt'])

    # ── 3. sum of net positions across all wallets per timestamp
    net_sum = (
        net_pos
        .groupby('dt')['net']
        .sum()
        .reset_index(name='total_net')
        .sort_values('dt')
    )

    fig = make_subplots(
        rows=1, cols=1,
        subplot_titles=['YES token (net: YES − NO position)'],
        specs=[[{'secondary_y': True}]],
    )

    COLORS = [
        '#4878CF', '#6ACC65', '#D65F5F', '#B47CC7', '#C4AD66',
        '#77BEDB', '#F7A35C', '#90ED7D', '#8085E9', '#F15C80',
    ]

    top_wallets = net_pos.groupby('wallet')['net'].agg(lambda x: x.abs().max()).sort_values(ascending=False).head(10).index

    # ── per-wallet net position (step: hold latest value, no interpolation)
    for c_idx, wallet in enumerate(top_wallets):
        wdf = net_pos[net_pos['wallet'] == wallet].sort_values('dt')
        fig.add_trace(
            go.Scatter(
                x=wdf['dt'],
                y=wdf['net'],
                mode='lines+markers',
                name=f'{wallet[:8]}…',
                line=dict(color=COLORS[c_idx % len(COLORS)], width=1, shape='hv'),
                marker=dict(size=4),
                legendgroup=wallet,
                opacity=0.6,
            ),
            row=1, col=1, secondary_y=False,
        )

    # ── total net position line (primary y, bold)
    fig.add_trace(
        go.Scatter(
            x=net_sum['dt'],
            y=net_sum['total_net'],
            mode='lines+markers',
            name='SUM (net YES)',
            line=dict(color='#222222', width=3, shape='hv'),
            marker=dict(size=5),
            legendgroup='sum_net',
        ),
        row=1, col=1, secondary_y=False,
    )

    # ── YES price line (secondary y)
    fig.add_trace(
        go.Scatter(
            x=price_ts['dt'],
            y=price_ts['vwap'],
            mode='lines+markers',
            name='Price (YES)',
            line=dict(color='#888888', width=2, dash='dot', shape='hv'),
            marker=dict(size=4),
            legendgroup='price_yes',
        ),
        row=1, col=1, secondary_y=True,
    )

    fig.update_yaxes(title_text='Net position (tokens)', row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text='Price (USDC)', row=1, col=1, secondary_y=True, range=[0, 1])

    fig.update_layout(
        title=f'Wallet net YES positions — {MARKET[:16]}…',
        height=500,
        template='plotly_white',
        legend=dict(orientation='v', x=1.05),
    )
    fig.show(renderer='browser')


In [591]:
# dominant tags
wallet_fills = df_full[df_full['wallet'].isin(wallet_cohorts['multi_filter']['wallet'])]

dominant_tags = (
    wallet_fills
    .groupby(['wallet', 'primary_tag'], as_index=False)[['quantity', 'pnl']].sum()
    .sort_values(['wallet', 'quantity'], ascending=[True, False])
    .assign(total_qty=lambda df: df.groupby('wallet')['quantity'].transform('sum'))
    .drop_duplicates('wallet')
    .assign(primary_tag_ratio=lambda df: df['quantity'] / df['total_qty'])
    .rename(columns={
        'quantity': 'primary_tag_qty'
    })[['wallet', 'primary_tag', 'primary_tag_qty', 'primary_tag_ratio', 'pnl']]
)

In [592]:
print(len(dominant_tags))
dominant_tags.sort_values('pnl', ascending=False).head(20)

218


,wallet,primary_tag,primary_tag_qty,primary_tag_ratio,pnl
3092,0xbacd00c9080a82ded56f504ee8810af732b0ab35,Politics,40341164.99,0.85,753227.34
3263,0xc6587b11a2209e46dfe3928b31c5514a8e33b784,Politics,28917854.41,0.81,607600.41
1417,0x44c1dfe43260c94ed4f1d00de2e1f80fb113ebc1,Politics,28225231.90,0.68,365496.16
2178,0x7c3db723f1d4d8cb9c550095203b686cb11e5c6b,Politics,46139775.94,0.79,337739.66
15,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,Politics,14688876.48,0.83,296656.48
1602,0x5739ddf8672627ce076eff5f444610a250075f1a,Politics,3231319.94,0.96,241552.61
1828,0x68c24bf4a8ad4d79a6fe4b8eec6f93a02dfd1711,Politics,5549638.78,0.88,236751.33
3642,0xdfe3fedc5c7679be42c3d393e99d4b55247b73c4,Politics,3184181.75,0.79,229387.86
3250,0xc4d1a863e9cc45d02ba22d3a1ae9ba7822018ce8,Politics,11365362.70,0.70,226598.27
2413,0x889e7f0464c72eb8cda1525ebc12b6aaba9d09e0,Politics,11130959.31,0.59,225386.89


In [593]:
# from polymarket_analysis.data.tags import load_tag_map
# from polymarket_analysis.data.tags import dominant_tag_stats

# split_dt = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

# _result = dominant_tag_stats(
#     df_trades=df_full[df_full['dt'] >= split_dt],
#     wallets=wallet_cohorts["multi_filter"]["wallet"],
# )

# print(f"Wallets: {len(_result)}")
# _result[_result['primary_tag'] == 'Politics'].head(5)


In [594]:
# _result.groupby('primary_tag').agg(
#     tag_pnl=('tag_pnl', 'sum'),
#     wallets=('wallet', 'nunique')
#     )

In [595]:
watched_wallets = wallet_cohorts['multi_filter'].sort_values("copyable_pnl", key=abs, ascending=False)['wallet'].head(10).to_list()
for w in watched_wallets:
    wallet_df = wallet_vol_train[wallet_vol_train['wallet'] == w]
    wallet_cohorts[w] = wallet_df.copy().reset_index(drop=True)

In [596]:
print(len(wallet_cohorts['multi_filter']))
pd.set_option('display.max_rows', 100)
wallet_cohorts["multi_filter"].sort_values("total_pnl", ascending=False).head(30)

218


,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top_market_pnl_pct,median_roi,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi
130,0xc6587b11a2209e46dfe3928b31c5514a8e33b784,10.90,1878.00,192.00,29029528.92,702646.16,92649.78,0.38,0.19,0.01,-0.04,91701.06,0.13,30140.95,0.33,0.02,0.13,-0.01
161,0xbacd00c9080a82ded56f504ee8810af732b0ab35,4.10,46103.00,1059.00,22196501.74,645638.20,140370.19,0.24,0.16,-1.00,-0.10,122370.76,0.19,77297.94,0.55,0.03,0.22,-0.02
102,0x889e7f0464c72eb8cda1525ebc12b6aaba9d09e0,9.29,1730.00,236.00,12957444.16,600861.06,3605.42,0.38,0.22,0.06,0.10,143594.34,0.24,50813.48,14.09,0.05,0.01,0.00
94,0x44c1dfe43260c94ed4f1d00de2e1f80fb113ebc1,7.27,5448.00,567.00,24075463.51,458591.40,78321.13,0.34,0.16,0.01,0.01,133275.67,0.29,96349.38,1.23,0.02,0.17,0.00
182,0x7c3db723f1d4d8cb9c550095203b686cb11e5c6b,4.02,30334.00,2076.00,27146811.52,318377.42,111467.32,0.29,0.15,0.00,-0.10,47413.59,0.15,31406.25,0.28,0.01,0.35,-0.04
117,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,3.89,3682.00,413.00,12386053.42,316581.20,36497.50,0.15,0.08,0.04,-0.03,43169.79,0.14,34643.70,0.95,0.03,0.12,-0.00
39,0x6bab41a0dc40d6dd4c1a915b8c01969479fd1292,4.69,3287.00,521.00,8600376.40,311067.18,90531.54,0.21,0.11,0.01,0.12,56515.67,0.18,19993.69,0.22,0.04,0.29,0.03
58,0xee50a31c3f5a7c77824b12a941a54388a2827ed6,9.26,651.00,30.00,953234.83,293613.57,8262.76,0.24,0.24,0.41,0.58,13550.70,0.05,2863.46,0.35,0.31,0.03,0.02
20,0xdfe3fedc5c7679be42c3d393e99d4b55247b73c4,10.74,1543.00,136.00,2301560.26,272752.75,52629.68,0.20,0.21,0.20,0.39,33000.00,0.12,2040.00,0.04,0.12,0.19,0.08
35,0x68c24bf4a8ad4d79a6fe4b8eec6f93a02dfd1711,10.28,4624.00,897.00,2517113.49,262084.12,54959.65,0.35,0.09,0.06,0.19,51778.22,0.20,21708.16,0.39,0.10,0.21,0.04


In [597]:
selected_wallets = wallet_cohorts["multi_filter"]
selected_wallets.to_parquet(WORKSPACE_DIR / "selected_wallets_v2.parquet", index=False)

## Build and save strategy registry

All cohorts × all trigger variants → persisted `WalletSelectionStrategy` files.
The backtest loads these directly.

In [599]:
# from polymarket_analysis.wallet_selection.selector import build_strategies_from_sweep
# from polymarket_analysis.strategy.registry import save_strategy

# all_strategies = build_strategies_from_sweep(
#     wallet_cohorts=wallet_cohorts,
#     signal_threshold=SIGNAL_THRESHOLD,
#     selection_metric=BEST_SELECTION_METRIC,
#     top_n=BEST_TOP_N,
#     sweep_df=cohort_sweep,
#     extra_metadata={
#         "end_date_train": str(END_DATE_TRAIN),
#         "train_a_end_date": str(TRAIN_A_END_DATE),
#     },
# )

# for strategy in all_strategies:
#     parquet_path, json_path = save_strategy(strategy, WORKSPACE_DIR)
#     print(f"Saved [{strategy.strategy_id}]  wallets={len(strategy.wallets):3d}  trigger={strategy.trigger.fn_ref.split('.')[-1]}")

# print(f"\nTotal strategies saved: {len(all_strategies)}")


## Strategy registry summary

In [600]:
# from polymarket_analysis.strategy.registry import load_all_strategies

# registry = load_all_strategies(WORKSPACE_DIR)
# summary_rows = []
# for sid, s in registry.items():
#     summary_rows.append({
#         "strategy_id": s.strategy_id,
#         "name": s.name,
#         "selection_method": s.selection_method,
#         "num_wallets": len(s.wallets),
#         "trigger_fn": s.trigger.fn_ref.split(".")[-1],
#         "threshold": s.trigger.params.get("threshold"),
#         "dynamic_sizing": s.trigger.params.get("dynamic_sizing"),
#     })

# pd.DataFrame(summary_rows)


## Wallet PnL over time

Three independent plots:
- **Training plot** — cohort aggregate cumulative PnL over the training period only
- **Test plot** — cohort aggregate cumulative PnL over the test period only (starts from 0)
- **Individual plot** — per-wallet cumulative PnL spanning train + test, with a train/test split
  vline and wallet address labels; test portion resets to 0 at the split boundary

Set `PLOT_WALLET_PNL = False` to skip (e.g. when running headlessly).


In [601]:
PLOT_WALLET_PNL  = True
TOP_N_INDIVIDUAL = 10   # wallets shown in panel 1 per cohort


In [602]:
df_fills = df_full.copy()
df_fills['copyable_pnl_exposure'] = np.where( 
    df_fills['copyable_qty'] > 0,
    df_fills['price'] * df_fills['copyable_qty'],
    np.nan
)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_fills.columns and "quantity" not in df_fills.columns:
    df_fills = df_fills.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_fills["usdc_amount"]      = df_fills["usdc_amount"].astype(float)
df_fills["final_value_usdc"] = df_fills["final_value_usdc"].astype(float)
df_fills["quantity"]         = df_fills["quantity"].astype(float)

split_dt = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

len(df_fills), len(df_fills[df_fills["dt"] < split_dt]), len(df_fills[df_fills["dt"] >= split_dt])

(4484112, 3519974, 964138)

In [603]:
markets = df_fills[(df_fills['wallet'].isin(wallet_cohorts['multi_filter']['wallet']))][['condition_id', 'tags', 'primary_tag']]
markets = markets.groupby('condition_id').agg(
    tags=('tags', 'first'),
    primary_tag=('primary_tag', 'first'),
).reset_index()
markets['has_mentions'] = markets['tags'].apply(lambda tags: 'Mentions' in tags)
mention_markets = set(markets[markets['has_mentions']]['condition_id'])
del markets
len(mention_markets)

7621

In [604]:
filter_fills = df_fills[
    (df_fills['wallet'].isin(wallet_cohorts['multi_filter']['wallet']))
    & (df_fills['side'] == 'BUY')
    & (df_fills['primary_tag'] == 'Politics')
    & (~df_fills['condition_id'].isin(mention_markets))
    & (df_fills["dt"] >= split_dt)
    ].copy().reset_index(drop=True)

print(len(filter_fills))
filter_fills = filter_fills[filter_fills['copyable_qty'] >= 1].copy().reset_index(drop=True)
print(len(filter_fills))
filter_fills.head()

80520
30613


,wallet,condition_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,trade_pnl,...,avail_copy_count,end_date_iso,question,tags,primary_tag,winner_token_id,outcome,pnl,notional,copyable_pnl_exposure
0,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0x09cbe3e796661a1d820580145488ad2ccb9ad1e720efcd64b448bb77b97007c1,2026-02-28 07:01:31+00:00,BUY,10000.00,10000.00,0.02,170.00,0.00,-170.00,...,263.00,2026-01-31T00:00:00Z,"US strikes Iran by February 27, 2026?","[Politics, Iran, Middle East, Geopolitics, World, Parent For Derivative, HFC]",Politics,20294480808593556060521766763233040937161644261106468868449852592476545850621,Yes,-170.00,170.00,170.00
1,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0x09cbe3e796661a1d820580145488ad2ccb9ad1e720efcd64b448bb77b97007c1,2026-02-28 07:02:01+00:00,BUY,63627.82,332.93,0.02,6.66,0.00,-6.66,...,287.00,2026-01-31T00:00:00Z,"US strikes Iran by February 27, 2026?","[Politics, Iran, Middle East, Geopolitics, World, Parent For Derivative, HFC]",Politics,20294480808593556060521766763233040937161644261106468868449852592476545850621,Yes,-6.66,6.66,6.66
2,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0x09cbe3e796661a1d820580145488ad2ccb9ad1e720efcd64b448bb77b97007c1,2026-02-28 07:02:01+00:00,BUY,63294.89,53294.89,0.02,1064.90,0.00,-1064.90,...,1704.00,2026-01-31T00:00:00Z,"US strikes Iran by February 27, 2026?","[Politics, Iran, Middle East, Geopolitics, World, Parent For Derivative, HFC]",Politics,20294480808593556060521766763233040937161644261106468868449852592476545850621,Yes,-1064.90,1064.90,1064.90
3,0x0a793254e040027c5d1ca0c2f87cda8e22228e05,0x09cbe3e796661a1d820580145488ad2ccb9ad1e720efcd64b448bb77b97007c1,2026-02-27 08:26:41+00:00,BUY,793.82,104.27,0.98,101.98,104.27,2.29,...,22.00,2026-01-31T00:00:00Z,"US strikes Iran by February 27, 2026?","[Politics, Iran, Middle East, Geopolitics, World, Parent For Derivative, HFC]",Politics,20294480808593556060521766763233040937161644261106468868449852592476545850621,No,2.29,101.98,101.98
4,0x0a793254e040027c5d1ca0c2f87cda8e22228e05,0x09cbe3e796661a1d820580145488ad2ccb9ad1e720efcd64b448bb77b97007c1,2026-02-27 08:26:49+00:00,BUY,837.11,43.29,0.98,42.34,43.29,0.95,...,22.00,2026-01-31T00:00:00Z,"US strikes Iran by February 27, 2026?","[Politics, Iran, Middle East, Geopolitics, World, Parent For Derivative, HFC]",Politics,20294480808593556060521766763233040937161644261106468868449852592476545850621,No,0.95,42.34,42.34


In [605]:
pd.set_option('display.max_colwidth', 100)

In [606]:
filter_fills["bucket"] = filter_fills["dt"].dt.floor('5min')

MAX_EXPOSURE = 100

df = filter_fills.groupby(['bucket', 'condition_id', 'wallet', 'side']).agg(
    copyable_pnl_exposure=('copyable_pnl_exposure', 'sum'),
    total_copyable_qty=('copyable_qty', 'sum'),
    trade_pnl =('trade_pnl', 'sum'),
    copyable_pnl = ('copyable_pnl', 'sum'),
    trades=('condition_id', 'count'),
    copyable_qty=('copyable_qty', 'sum'),
    wallets=('wallet', lambda x: x.nunique()),
).reset_index()

scale = np.minimum(1, MAX_EXPOSURE / df["copyable_pnl_exposure"].replace(0, np.nan))

df = df.assign(
    scaled_copyable_pnl_exposure = df["copyable_pnl_exposure"] * scale,
    scaled_copyable_pnl = df["copyable_pnl"] * scale,
    scaled_copyable_qty = df["copyable_qty"] * scale,
)

df.head(1)

,bucket,condition_id,wallet,side,copyable_pnl_exposure,total_copyable_qty,trade_pnl,copyable_pnl,trades,copyable_qty,wallets,scaled_copyable_pnl_exposure,scaled_copyable_pnl,scaled_copyable_qty
0,2026-02-16 00:25:00+00:00,0x9d52a1ad771507479cb17d8748705f5d0556796669fd7dec42e6766400a8f445,0xbacd00c9080a82ded56f504ee8810af732b0ab35,BUY,56.50,100.00,-56.50,-56.50,1,100.00,1,56.50,-56.50,100.00


In [607]:
df = (df.merge(mdf, on='condition_id', how='inner')
    .sort_values(['bucket', 'copyable_pnl_exposure'], ascending=[True, False])
    .reset_index(drop=True)
)
df['end_date_iso'] = pd.to_datetime(df['end_date_iso'], utc=True)
df = df[df['end_date_iso'] > split_dt].copy().reset_index(drop=True)

In [608]:
df_plot = df.groupby('end_date_iso').agg(
    scaled_copyable_pnl=('scaled_copyable_pnl', 'sum'),
    ending_exposure=('scaled_copyable_pnl_exposure', 'sum'),
).reset_index()

df.sort_values('bucket', inplace=True)
df['scaled_copyable_pnl_cumsum'] = df['scaled_copyable_pnl'].cumsum()

df_pnl = df[['bucket', 'scaled_copyable_pnl_cumsum']].copy()

# exposure ends at EOD of end_date, so shift by 24h
df_exposure = df_plot[['end_date_iso', 'ending_exposure']].copy()
df_exposure['end_date_iso'] = pd.to_datetime(df_exposure['end_date_iso']) + pd.Timedelta(days=1)
df_exposure.rename(columns={'end_date_iso': 'dt', 'ending_exposure': 'exposure_delta'}, inplace=True)

resolution_exposure = df_exposure.groupby('dt').agg(exposure_delta=('exposure_delta', 'sum')).reset_index()
resolution_exposure['exposure_delta'] = resolution_exposure['exposure_delta'] * -1

buy_exposure = df[['bucket', 'scaled_copyable_pnl_exposure']].rename(columns={'bucket': 'dt', 'scaled_copyable_pnl_exposure': 'exposure_delta'})

df_exposure = ( pd.concat([resolution_exposure, buy_exposure], ignore_index=True)
    .reset_index(drop=True)
)

df_exposure.sort_values('dt', inplace=True)
df_exposure['exposure'] = df_exposure['exposure_delta'].cumsum()

# df_plot['end_date_iso'] = pd.to_datetime(df_plot['end_date_iso']) + pd.Timedelta(days=1)

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_exposure['dt'],
    y=df_exposure['exposure'],
    mode='lines',
    name='exposure'
))

fig.add_trace(go.Scatter(
    x=df_pnl['bucket'],
    y=df_pnl['scaled_copyable_pnl_cumsum'],
    mode='lines',
    name='scaled_copyable_pnl_cumsum'
))

fig.show()

In [609]:
import plotly.express as px

# Aggregate by bucket
bucket_pnl = df.groupby('bucket')['scaled_copyable_pnl'].sum().reset_index()
bucket_pnl['scaled_copyable_pnl'] = bucket_pnl['scaled_copyable_pnl'].cumsum()

# Plot with Plotly
fig = px.line(
    bucket_pnl,
    x='bucket',
    y='scaled_copyable_pnl',
    title='Total Scaled Copyable PnL by Time Bucket',
    labels={'bucket': 'Time Bucket', 'scaled_copyable_pnl': 'Scaled Copyable PnL'}
)
fig.show()

In [610]:
( 
    df.groupby('condition_id').agg(
        total_copyable_exposure=('copyable_pnl_exposure', 'sum'),
        total_copyable_qty=('copyable_qty', 'sum'),
        trade_pnl =('trade_pnl', 'sum'),
        copyable_pnl = ('copyable_pnl', 'sum'),
        trades=('condition_id', 'count'),
        wallets=('wallet', lambda x: x.nunique()),
 ).sort_values('copyable_pnl', ascending=False).quantile([0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99])
)

,total_copyable_exposure,total_copyable_qty,trade_pnl,copyable_pnl,trades,wallets
0.01,0.05,1.64,-3008.50,-1544.51,1.00,1.00
0.10,0.70,15.73,-78.10,-61.65,1.00,1.00
0.20,4.21,40.83,-10.96,-11.39,2.00,1.00
0.30,17.45,97.70,-2.38,-2.85,2.00,2.00
0.40,41.01,175.62,-0.55,-0.78,4.00,2.00
0.50,110.30,320.27,0.49,-0.08,6.00,3.00
0.60,241.78,662.94,7.62,1.45,8.00,3.00
0.70,539.89,1150.54,30.76,7.65,15.00,4.10
0.80,1570.25,2722.79,155.92,40.19,24.00,6.00
0.90,5396.26,8720.76,643.46,220.37,48.70,9.00


In [616]:
from polymarket_analysis.visualization.wallet_plots import (
    plot_wallet_selection_pnl,
    plot_wallet_individual_pnl,
)

if PLOT_WALLET_PNL:
    # Cohort aggregate PnL — training period
    fig_train = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        period="train",
        title="Wallet selection cohorts — cohort cumulative PnL (training period)",
    )
    fig_train.show(renderer="browser")

    # Cohort aggregate PnL — test period (starts from 0)
    fig_test = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        period="test",
        title="Wallet selection cohorts — cohort cumulative PnL (test period)",
    )
    fig_test.show(renderer="browser")

    # # Individual wallet lines (train + test) with split vline and labels
    # fig_ind = plot_wallet_individual_pnl(
    #     df_fills,
    #     wallet_cohorts,
    #     split_date=split_dt,
    #     top_n_individual=TOP_N_INDIVIDUAL,
    #     title="Individual wallet cumulative PnL (train + test, resets at split)",
    # )
    # fig_ind.show(renderer="browser")

else:
    print("PLOT_WALLET_PNL=False — skipping plots")